# London's Blue Plaques — who gets remembered, and where

English Heritage runs the famous blue plaque scheme: ~1,036 markers across London,
each pinning a person to a building. This notebook scrapes them all and asks a
simple question — *who* does London choose to remember, and *where* does that
memory physically live?

**Data:** scraped from the [English Heritage blue plaques](https://www.english-heritage.org.uk/visit/blue-plaques/)
search API + detail pages (see `scrape_plaques.py`). Snapshot: July 2026.

In [1]:
import re
from collections import Counter

import gender_guesser.detector as gender
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/blue_plaques_2026-07.csv")
print(df.shape)
df.head(3)

(1036, 16)


,id,name,born,died,professions,address,borough,summary,detail_url,profession_detail,category,inscription,material,lat,lng,biography
0,15012,14 BUCKINGHAM STREET,NaN,NaN,"Statesman, Diarist, Naval Official, Painter","14 Buckingham Street, Charing Cross, London, W...",City Of Westminster,Blue Plaque commemorating diarist Samuel Pepys...,/visit/blue-plaques/buckingham-street/,"Statesman, Diarist, Naval Official, Painter","Fine Arts, Literature, Politics and Administra...",In a House formerly standing on this site live...,Encaustic with border tiles,51.508240,-0.123184,NaN
1,172752,32 SOHO SQUARE,NaN,NaN,Botanists,"32 Soho Square, Soho, London, W1D 3AP, City Of...",City Of Westminster,Blue plaque commemorating botanists Sir Joseph...,/visit/blue-plaques/soho-square/,Botanists,"Historical Sites, Science",SIR JOSEPH BANKS 1743-1820 PRESIDENT OF THE RO...,Stone,51.514740,-0.132576,"The botanists Sir Joseph Banks, Robert Brown a..."
2,87558,Sir Patrick Abercrombie,1879.0,1957.0,Town and country planner,"Flat 1, 63 Egerton Gardens, Brompton, SW3 2BZ,...",London Borough of Kensington And Chelsea,Blue plaque commemorating the pioneer of town ...,/visit/blue-plaques/sir-patrick-abercrombie/,Town and country planner,Architecture and Building,Sir PATRICK ABERCROMBIE 1879-1957 Pioneer of t...,Ceramic,51.496588,-0.168453,Sir Patrick Abercrombie is the most celebrated...


## Clean and derive

Shorten borough names, flag person-plaques (those with a death year), compute age at death, and take each plaque's primary category.

In [2]:
df["borough_short"] = (
    df["borough"].astype(str)
    .str.replace("London Borough of ", "", regex=False)
    .str.replace("Royal Borough of ", "", regex=False)
    .str.replace("City Of ", "", regex=False)
    .str.strip()
)
df["is_person"] = df["died"].notna()
df["age_at_death"] = df["died"] - df["born"]
df["category_main"] = df["category"].astype(str).str.split(",").str[0].str.strip()
df[["name", "born", "died", "borough_short", "category_main"]].head()

,name,born,died,borough_short,category_main
0,14 BUCKINGHAM STREET,NaN,NaN,Westminster,Fine Arts
1,32 SOHO SQUARE,NaN,NaN,Westminster,Historical Sites
2,Sir Patrick Abercrombie,1879.0,1957.0,Kensington And Chelsea,Architecture and Building
3,Harold Abrahams,1899.0,1978.0,Barnet,Sport
4,"ROBERT and HOOD, THOMAS and GALSWORTHY, JOHN a...",NaN,NaN,Westminster,Applied Arts


## How complete is the data?

Before trusting any angle, check field coverage. Anything near 100% is safe to
lean on; low-coverage fields get used lightly or not at all.

In [3]:
coverage = (1 - df.isna().mean()).round(3).sort_values(ascending=False)
coverage.to_frame("non_null_share")

,non_null_share
id,1.000
is_person,1.000
borough_short,1.000
address,1.000
borough,1.000
summary,1.000
detail_url,1.000
name,1.000
inscription,1.000
lng,0.999


Name, borough, coordinates, inscription and category are all ~99–100%. Birth/death
years cover 96% (990 people). Only `biography` is patchy (41%), so we avoid it.

There is **no plaque erection date** anywhere on the site, so "how long after death
did recognition arrive?" simply can't be answered from this source — an honest dead end.

## Inferring gender

There's no gender field, so we infer it from first names (offline `gender-guesser`) plus honorifics (Sir/Dame/Lady). Imperfect — non-Western names and initials-only entries stay *unknown* — so we report the female share **of resolved names** and keep the caveat visible.

In [4]:
_DET = gender.Detector(case_sensitive=False)
_HON = set("sir dame dr lord lady rev prof mrs mr miss ms sr st captain major "
           "general admiral the rt hon".split())


def first_name(name):
    toks = [t for t in str(name).replace(".", "").split() if t.lower() not in _HON]
    return toks[0] if toks else None


def infer_gender(name):
    low = f" {str(name).lower()} "
    if any(h in low for h in (" dame ", " lady ", " mrs ", " miss ")):
        return "female"
    if any(h in low for h in (" sir ", " lord ")):
        return "male"
    fn = first_name(name)
    if not fn:
        return "unknown"
    g = _DET.get_gender(fn)
    if g in ("male", "mostly_male"):
        return "male"
    if g in ("female", "mostly_female"):
        return "female"
    return "unknown"


person = df[df["is_person"]].copy()
person["gender"] = person["name"].map(infer_gender)
counts = person["gender"].value_counts()
resolved = person[person["gender"].isin(["male", "female"])]
female_share = (resolved["gender"] == "female").mean() * 100
print(counts.to_dict())
print(f"Women: {female_share:.1f}% of resolved names")

{'male': 716, 'female': 152, 'unknown': 122}
Women: 17.5% of resolved names


## The map: where London's memory lives

Every plaque with coordinates, coloured by its main category. Zoom in — the density in the centre and west is immediate.

In [5]:
geo = df.dropna(subset=["lat", "lng"]).copy()
top_cats = geo["category_main"].value_counts().head(10).index
geo["cat_plot"] = geo["category_main"].where(geo["category_main"].isin(top_cats), "Other")

fig_map = px.scatter_map(
    geo, lat="lat", lon="lng", hover_name="name",
    hover_data={"borough_short": True, "category_main": True, "lat": False, "lng": False},
    color="cat_plot", zoom=10.2, height=640,
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig_map.update_layout(map_style="carto-positron",
                      margin={"r": 0, "t": 10, "l": 0, "b": 0},
                      legend_title_text="Category")
fig_map.show()

## Three boroughs hold two-thirds of the memory

In [6]:
top3 = df["borough_short"].value_counts().head(3)
print(f"Top-3 borough share: {top3.sum() / len(df) * 100:.1f}%")
bor = df["borough_short"].value_counts().head(15).sort_values()
fig_bor = px.bar(x=bor.values, y=bor.index, orientation="h",
                 labels={"x": "Plaques", "y": ""}, height=520,
                 title="Blue plaques by borough (top 15)")
fig_bor.show()

Top-3 borough share: 68.6%


## Who gets remembered: the professions of the dead

In [7]:
cat = df["category_main"].value_counts().head(15).sort_values()
fig_cat = px.bar(x=cat.values, y=cat.index, orientation="h",
                 labels={"x": "Plaques", "y": ""}, height=520,
                 title="Plaques by primary category (top 15)")
fig_cat.show()

## The gender gap over time

Stacked by birth decade. The blue wall is hard to miss — but watch the recent decades.

In [8]:
kn = resolved.dropna(subset=["born"]).copy()
kn["decade"] = (kn["born"] // 10 * 10).astype(int)
gt = kn.groupby(["decade", "gender"]).size().unstack(fill_value=0)
gt = gt[gt.sum(axis=1) >= 5]
fig_gen = px.bar(gt, barmode="stack",
                 labels={"value": "People commemorated", "decade": "Birth decade"},
                 height=460, title="Commemorated people by birth decade and inferred gender",
                 color_discrete_map={"female": "#c2185b", "male": "#1565c0"})
fig_gen.show()

## Lifespans

How long did the commemorated live? (Filtered to plausible 0–110 years.)

In [9]:
life = person.dropna(subset=["age_at_death"])
life = life[(life["age_at_death"] > 0) & (life["age_at_death"] < 110)]
print("median age at death:", life["age_at_death"].median())
fig_life = px.histogram(life, x="age_at_death", nbins=40,
                        labels={"age_at_death": "Age at death"}, height=430,
                        title="Age at death of the commemorated")
fig_life.show()

median age at death: 72.0


## What the plaques actually say

The most common words across all 1,036 inscriptions (stopwords removed).

In [10]:
STOP = set('''a an the of and to in on at for with by from as is was were are be been being
his her he she who whom which that this these those first here lived worked born died
home house where one two blue plaque commemorating'''.split())
words = Counter()
for text in df["inscription"].dropna():
    for w in re.findall(r"[a-zA-Z]{3,}", str(text).lower()):
        if w not in STOP:
            words[w] += 1
top_words = pd.Series(dict(words.most_common(25))).sort_values()
fig_words = px.bar(x=top_words.values, y=top_words.index, orientation="h",
                   labels={"x": "Times used", "y": ""}, height=560,
                   title="Most common words in blue plaque inscriptions")
fig_words.show()

## Takeaways

- **London's memory is geographically tiny.** Just three boroughs — Westminster,
  Kensington & Chelsea and Camden — hold roughly **two-thirds** of every blue plaque.
- **It skews overwhelmingly male.** Fewer than **1 in 5** commemorated people (of
  resolved names) are women, though the most recent decades are less lopsided.
- **It is a city of writers.** Literature is the single largest category, and the
  most common inscription words are *poet, writer, painter, novelist, artist* — plus
  *Sir*, a quiet marker of the honours-era men who dominate the wall.

In [11]:
from pathlib import Path
EXPORTS = Path("exports"); EXPORTS.mkdir(exist_ok=True)
fig_map.write_html(EXPORTS / "plaque-map.html", include_plotlyjs="cdn")
fig_bor.write_html(EXPORTS / "by-borough.html", include_plotlyjs="cdn")
fig_cat.write_html(EXPORTS / "by-category.html", include_plotlyjs="cdn")
fig_gen.write_html(EXPORTS / "gender-by-decade.html", include_plotlyjs="cdn")
fig_life.write_html(EXPORTS / "lifespans.html", include_plotlyjs="cdn")
fig_words.write_html(EXPORTS / "inscription-words.html", include_plotlyjs="cdn")
print("exports written:", [p.name for p in sorted(EXPORTS.glob("*.html"))])

exports written: ['by-borough.html', 'by-category.html', 'gender-by-decade.html', 'inscription-words.html', 'lifespans.html', 'plaque-map.html']
